# ***`Feature Engineering Notebook`***
### ***Notebook 02 – Creating Modeling-Ready Features***

## ***1. Objective:***
This notebook focuses on preparing advanced features for machine learning models.

✅ Handle categorical variables  
✅ Create interaction terms  
✅ Create numerical transformations  
✅ Prepare lag features (if needed for time series with ML)  
✅ Export final feature-engineered dataset  

This notebook acts as a bridge between EDA and ML Pipeline.

## ***2. Importing Required Libraries***

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## ***3. Loading Cleaned Dataset***

In [5]:
df = pd.read_csv(r"C:\Users\tejas\OneDrive\Desktop\CPMLF Capstone Projects\CREDIT CARD RISK\data\processed\loan_data_cleaned.csv")

In [6]:
df

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,addr_state:UT,addr_state:VA,addr_state:VT,addr_state:WA,addr_state:WI,addr_state:WV,addr_state:WY,initial_list_status:f,initial_list_status:w,good_bad
0,1077501,1296599,5000,5000,4975.0,0,10.65,162.87,1,6,...,0,0,0,0,0,0,0,1,0,1
1,1077430,1314167,2500,2500,2500.0,1,15.27,59.83,2,13,...,0,0,0,0,0,0,0,1,0,0
2,1077175,1313524,2400,2400,2400.0,0,15.96,84.33,2,14,...,0,0,0,0,0,0,0,1,0,1
3,1076863,1277178,10000,10000,10000.0,0,13.49,339.31,2,10,...,0,0,0,0,0,0,0,1,0,1
4,1075358,1311748,3000,3000,3000.0,1,12.69,67.79,1,9,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466280,8598660,1440975,18400,18400,18400.0,1,14.47,432.64,2,11,...,0,0,0,0,0,0,0,0,1,1
466281,9684700,11536848,22000,22000,22000.0,1,19.97,582.50,3,19,...,0,0,0,0,0,0,0,1,0,0
466282,9584776,11436914,20700,20700,20700.0,1,16.99,514.34,3,15,...,0,0,0,0,0,0,0,1,0,1
466283,9604874,11457002,2000,2000,2000.0,0,7.90,62.59,0,3,...,0,0,0,0,0,0,0,0,1,1


In [8]:
df_fe = df.copy()

## ***3. Drop Leakage Columns***

In [9]:
leakage_cols = [
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt",
    "next_pymnt_d", "last_credit_pull_d",
    "loan_status"
]

cols_to_drop = [c for c in leakage_cols if c in df_fe.columns]

df_fe.drop(columns=cols_to_drop, inplace=True)

## ***4. Basic Numeric Transforms***

### ***Log transform skewed money columns***

In [10]:
money_cols = [
    "loan_amnt", "funded_amnt", "installment",
    "annual_inc", "revol_bal", "tot_cur_bal"
]

for col in money_cols:
    if col in df_fe.columns:
        df_fe[f"log_{col}"] = np.log1p(df_fe[col])

## ***5. Credit Behaviour Features***

### ***5.1 EMI Burdens***

In [11]:
df_fe["monthly_income"] = df_fe["annual_inc"] / 12
df_fe["emi_to_income"] = df_fe["installment"] / df_fe["monthly_income"]

### ***5.2 Credit Utilization Stability***

In [12]:
df_fe["revol_util_frac"] = df_fe["revol_util"] / 100

### ***5.3 Account Load***

In [13]:
df_fe["acc_utilization"] = df_fe["open_acc"] / df_fe["total_acc"]
df_fe["acc_utilization"] = df_fe["acc_utilization"].clip(0, 1)

## ***6. Credit History Strength***

In [14]:
df_fe["credit_age_years"] = df_fe["mths_since_earliest_cr_line"] / 12

## ***7. Delinquecy Engineering***

In [15]:
df_fe["has_delinquency"] = (df_fe["delinq_2yrs"] > 0).astype(int)
df_fe["has_public_record"] = (df_fe["pub_rec"] > 0).astype(int)

## ***8. Employment Stability***

In [16]:
df_fe["emp_length_bucket"] = pd.cut(
    df_fe["emp_length_int"],
    bins=[-1, 1, 3, 5, 10, 50],
    labels=["very_low", "low", "mid", "high", "very_high"]
)

## ***8. Term plus Rate Interaction***

In [17]:
df_fe["rate_per_year"] = df_fe["int_rate"] * df_fe["term_int"] / 12

## ***9. Dropping Raw Redundant Cols***

In [18]:
drop_raw = [
    "emp_title",
    "desc",
    "title",
    "url",
    "zip_code",
    "addr_state",
    "earliest_cr_line",
    "issue_d"
]

df_fe.drop(columns=[c for c in drop_raw if c in df_fe.columns], inplace=True)

## ***10. Final Last Check***

In [19]:
df_fe.isnull().mean().sort_values(ascending=False).head(10)

verification_status_joint    1.0
mths_since_rcnt_il           1.0
open_il_12m                  1.0
open_il_6m                   1.0
open_acc_6m                  1.0
total_bal_il                 1.0
il_util                      1.0
dti_joint                    1.0
annual_inc_joint             1.0
open_rv_12m                  1.0
dtype: float64

In [20]:
df_fe.shape

(466285, 202)

## ***11. Saving ML Ready Dataset***

In [23]:
df_fe.to_csv(r"C:\Users\tejas\OneDrive\Desktop\CPMLF Capstone Projects\CREDIT CARD RISK\data\processed\loan_data_feature_engineered.csv", index=False)

In [25]:
df_fe

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,log_tot_cur_bal,monthly_income,emi_to_income,revol_util_frac,acc_utilization,credit_age_years,has_delinquency,has_public_record,emp_length_bucket,rate_per_year
0,1077501,1296599,5000,5000,4975.0,0,10.65,162.87,1,6,...,NaN,2000.000000,0.081435,0.837,0.333333,32.916667,0,0,high,31.95
1,1077430,1314167,2500,2500,2500.0,1,15.27,59.83,2,13,...,NaN,2500.000000,0.023932,0.094,0.750000,18.666667,0,0,very_low,76.35
2,1077175,1313524,2400,2400,2400.0,0,15.96,84.33,2,14,...,NaN,1021.000000,0.082595,0.985,0.200000,16.083333,0,0,high,47.88
3,1076863,1277178,10000,10000,10000.0,0,13.49,339.31,2,10,...,NaN,4100.000000,0.082759,0.210,0.270270,21.833333,0,0,high,40.47
4,1075358,1311748,3000,3000,3000.0,1,12.69,67.79,1,9,...,NaN,6666.666667,0.010169,0.539,0.394737,21.916667,0,0,very_low,63.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
466280,8598660,1440975,18400,18400,18400.0,1,14.47,432.64,2,11,...,12.594727,9166.666667,0.047197,0.776,0.500000,14.666667,0,0,mid,72.35
466281,9684700,11536848,22000,22000,22000.0,1,19.97,582.50,3,19,...,12.309671,6500.000000,0.089615,0.463,0.600000,20.500000,0,1,high,99.85
466282,9584776,11436914,20700,20700,20700.0,1,16.99,514.34,3,15,...,11.206387,3833.333333,0.134176,0.511,0.418605,16.000000,0,0,high,84.95
466283,9604874,11457002,2000,2000,2000.0,0,7.90,62.59,0,3,...,13.290605,6916.666667,0.009049,0.215,0.777778,14.833333,1,0,low,23.70
